# EEG Dataset Inspection

This notebook performs **data inspection only** for ADFTD, BrainLat, and AD-Auditory. It does not preprocess, normalize, rename, move, delete, combine, or split any recordings.

The code first looks for the requested project layout (`data/ADFTD`, `data/BrainLat`, `data/AD_Auditory`). For the current workspace it also recognizes the downloaded source layout: `SIH_Data/dt1`, `SIH_Data/dt 2`, and `SIH_Data/dt 3`. BrainLat is restricted to AD, bvFTD/FTD, and healthy-control records; PD and MS are reported as excluded and are never included in the analysis tables.

In [23]:
# Install compatible inspection dependencies in the active notebook kernel when needed.
%pip install --force-reinstall "numpy==1.24.4" "pandas==1.5.3" "mne==1.6.1"

  Using cached numpy-1.24.4-cp39-cp39-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (17.3 MB)
  Using cached pandas-1.5.3-cp39-cp39-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (12.2 MB)
  Using cached mne-1.6.1-py3-none-any.whl (8.3 MB)
  Using cached pytz-2026.3.post1-py2.py3-none-any.whl (508 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl (229 kB)
  Using cached tqdm-4.70.0-py3-none-any.whl (80 kB)
  Using cached matplotlib-3.9.4-cp39-cp39-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (8.3 MB)
  Using cached scipy-1.13.1-cp39-cp39-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (38.6 MB)
  Using cached lazy_loader-0.5-py3-none-any.whl (8.0 kB)
  Using cached packaging-26.3-py3-none-any.whl (129 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
  Using cached decorator-5.3.1-py3-none-any.whl (10 kB)
  Using cached pooch-1.9.0-py3-none-any.whl (67 kB)
  Using cached importlib_resources-6.5.2-py3-none-any.whl (37 kB)
  Using cached pillow-11.3.0-cp39-cp3

In [1]:
from pathlib import Path
import json
import os
import re
from collections import Counter

import numpy as np
import pandas as pd

try:
    import mne
except ImportError as exc:
    raise ImportError("Install mne before running this notebook: pip install mne") from exc

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 160)

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != "NOTEBOOK":
    NOTEBOOK_DIR = Path(".").resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "NOTEBOOK" else NOTEBOOK_DIR

DATA_CANDIDATES = {
    "ADFTD": [PROJECT_ROOT / "data" / "ADFTD", PROJECT_ROOT / "SIH_Data" / "dt1"],
    "BrainLat": [PROJECT_ROOT / "data" / "BrainLat", PROJECT_ROOT / "SIH_Data" / "dt 2"],
    "AD_Auditory": [PROJECT_ROOT / "data" / "AD_Auditory", PROJECT_ROOT / "SIH_Data" / "dt 3"],
}

def first_existing(paths):
    return next((path for path in paths if path.exists()), paths[0])

DATA_ROOTS = {name: first_existing(paths) for name, paths in DATA_CANDIDATES.items()}
print("Project root:", PROJECT_ROOT)
for name, path in DATA_ROOTS.items():
    print(f"{name}: {path} | exists={path.exists()}")

Project root: /home/admin1/Desktop/SIH_2026_ALL
ADFTD: /home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1 | exists=True
BrainLat: /home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt 2 | exists=True
AD_Auditory: /home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt 3 | exists=True


## 1. Inventory directories and metadata

This section inventories files without opening EEG signal arrays. It records common metadata formats and all EEG-like extensions so missing or unexpected files are visible before any header inspection.

In [15]:
EEG_EXTENSIONS = {".edf", ".fif", ".fif.gz", ".bdf", ".set", ".vhdr", ".eeg", ".cnt", ".mff", ".con", ".mat"}
COMPANION_EXTENSIONS = {".fdt", ".dat"}
METADATA_EXTENSIONS = {".csv", ".tsv", ".json", ".txt", ".xlsx", ".yaml", ".yml"}
PRIMARY_EXTENSIONS = EEG_EXTENSIONS - {".eeg", ".mat"}

def inventory_dataset(root):
    files = list(root.rglob("*")) if root.exists() else []
    regular = [path for path in files if path.is_file()]
    return {
        "all_files": regular,
        "metadata_files": sorted([p for p in regular if p.suffix.lower() in METADATA_EXTENSIONS]),
        "eeg_files": sorted([p for p in regular if p.suffix.lower() in EEG_EXTENSIONS or p.name.lower().endswith(".fif.gz")]),
        "companion_files": sorted([p for p in regular if p.suffix.lower() in COMPANION_EXTENSIONS]),
    }

INVENTORIES = {name: inventory_dataset(root) for name, root in DATA_ROOTS.items()}
inventory_rows = []
for name, inventory in INVENTORIES.items():
    inventory_rows.append({
        "Dataset": name,
        "Root": str(DATA_ROOTS[name]),
        "Total files": len(inventory["all_files"]),
        "EEG-like files": len(inventory["eeg_files"]),
        "Metadata files": len(inventory["metadata_files"]),
        "Extensions": ", ".join(sorted({p.suffix.lower() for p in inventory["eeg_files"]})),
    })
inventory_table = pd.DataFrame(inventory_rows)
display(inventory_table)

for name, inventory in INVENTORIES.items():
    print(f"\n{name} metadata files:")
    for path in inventory["metadata_files"]:
        print(" -", path.relative_to(DATA_ROOTS[name]))

,Dataset,Root,Total files,EEG-like files,Metadata files,Extensions
0,ADFTD,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1,357,176,179,.set
1,BrainLat,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt 2,145,100,13,.set
2,AD_Auditory,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt 3,183,35,109,.set



ADFTD metadata files:
 - dataset_description.json
 - participants.json
 - participants.tsv
 - sub-001/eeg/sub-001_task-eyesclosed_channels.tsv
 - sub-001/eeg/sub-001_task-eyesclosed_eeg.json
 - sub-002/eeg/sub-002_task-eyesclosed_channels.tsv
 - sub-002/eeg/sub-002_task-eyesclosed_eeg.json
 - sub-003/eeg/sub-003_task-eyesclosed_channels.tsv
 - sub-003/eeg/sub-003_task-eyesclosed_eeg.json
 - sub-004/eeg/sub-004_task-eyesclosed_channels.tsv
 - sub-004/eeg/sub-004_task-eyesclosed_eeg.json
 - sub-005/eeg/sub-005_task-eyesclosed_channels.tsv
 - sub-005/eeg/sub-005_task-eyesclosed_eeg.json
 - sub-006/eeg/sub-006_task-eyesclosed_channels.tsv
 - sub-006/eeg/sub-006_task-eyesclosed_eeg.json
 - sub-007/eeg/sub-007_task-eyesclosed_channels.tsv
 - sub-007/eeg/sub-007_task-eyesclosed_eeg.json
 - sub-008/eeg/sub-008_task-eyesclosed_channels.tsv
 - sub-008/eeg/sub-008_task-eyesclosed_eeg.json
 - sub-009/eeg/sub-009_task-eyesclosed_channels.tsv
 - sub-009/eeg/sub-009_task-eyesclosed_eeg.json
 - sub-0

## 2. Load labels only when supported by metadata

Labels are never inferred from arbitrary filenames. The mappings below use the downloaded metadata: ADFTD `Group`, BrainLat diagnosis/records CSVs, and AD-Auditory `Group`. Any value that cannot be mapped is retained as `Unknown` and listed for review.

In [16]:
def read_table(path):
    try:
        return pd.read_csv(path, sep="\t" if path.suffix.lower() == ".tsv" else ",", dtype=str)
    except Exception as exc:
        print(f"Could not read metadata {path}: {exc}")
        return pd.DataFrame()

def load_participants(root):
    for filename in ("participants.tsv", "participants.csv"):
        path = root / filename
        if path.exists():
            return read_table(path), path
    return pd.DataFrame(), None

def normalise_label(dataset, value):
    value = str(value).strip() if pd.notna(value) else ""
    if dataset == "ADFTD":
        return {"A": "AD", "F": "FTD", "C": "HC"}.get(value, "Unknown")
    if dataset == "AD_Auditory":
        return {"Mild AD": "AD", "AD": "AD", "Normal": "HC", "HC": "HC", "MCI": "MCI"}.get(value, "Unknown")
    return {"AD": "AD", "FTD": "FTD", "bvFTD": "FTD", "CN": "HC", "HC": "HC"}.get(value, "Unknown")

participant_tables = {}
label_tables = {}
for dataset in ("ADFTD", "AD_Auditory"):
    table, source = load_participants(DATA_ROOTS[dataset])
    participant_tables[dataset] = table
    if not table.empty and "participant_id" in table.columns and "Group" in table.columns:
        labels = table[["participant_id", "Group"]].copy()
        labels["Subject_ID"] = labels["participant_id"]
        labels["Diagnosis"] = labels["Group"].map(lambda value: normalise_label(dataset, value))
        label_tables[dataset] = labels[["Subject_ID", "Diagnosis", "Group"]]
    else:
        label_tables[dataset] = pd.DataFrame(columns=["Subject_ID", "Diagnosis", "Group"])
    print(f"{dataset}: participant metadata = {source}")

brainlat_root = DATA_ROOTS["BrainLat"]
brainlat_demographics = next(iter(sorted(brainlat_root.rglob("Demographics*.csv"))), None)
brainlat_records = next(iter(sorted(brainlat_root.rglob("Records*.csv"))), None)
if brainlat_demographics is not None:
    demographic_parts = [read_table(path) for path in sorted(brainlat_root.rglob("Demographics*.csv"))]
    brainlat_labels = pd.concat(demographic_parts, ignore_index=True) if demographic_parts else pd.DataFrame()
else:
    brainlat_labels = pd.DataFrame()
if not brainlat_labels.empty and {"id EEG", "diagnosis"}.issubset(brainlat_labels.columns):
    label_tables["BrainLat"] = brainlat_labels.rename(columns={"id EEG": "Subject_ID"})[["Subject_ID", "diagnosis"]].rename(columns={"diagnosis": "Source_Label"})
    label_tables["BrainLat"]["Diagnosis"] = label_tables["BrainLat"]["Source_Label"].map(lambda value: normalise_label("BrainLat", value))
else:
    label_tables["BrainLat"] = pd.DataFrame(columns=["Subject_ID", "Source_Label", "Diagnosis"])

print("BrainLat diagnosis source:", brainlat_demographics)
print("BrainLat records source:", brainlat_records)
for dataset, labels in label_tables.items():
    print(f"\n{dataset} source labels:")
    display(labels["Diagnosis"].value_counts(dropna=False).rename_axis("Diagnosis").reset_index(name="Subjects"))

ADFTD: participant metadata = /home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/participants.tsv
AD_Auditory: participant metadata = /home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt 3/participants.tsv
BrainLat diagnosis source: /home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt 2/dt 2/Demographics_AD_EEG_data.csv
BrainLat records source: /home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt 2/dt 2/Records_AD_EEG_data.csv

ADFTD source labels:


,Diagnosis,Subjects
0,AD,36
1,HC,29
2,FTD,23



AD_Auditory source labels:


,Diagnosis,Subjects
0,AD,16
1,HC,10
2,MCI,6
3,Unknown,3



BrainLat source labels:


,Diagnosis,Subjects
0,HC,46
1,AD,35
2,FTD,19


## 3. Build the recording inventory

EEGLAB `.fdt` files are companion data files and are not counted as separate recordings when a matching `.set` file exists. BrainLat is filtered to the metadata-supported AD, FTD/bvFTD, and HC groups only. A missing metadata match remains in the table with `Unknown` rather than receiving a guessed label.

In [17]:
def subject_from_filename(dataset, path):
    match = re.search(r"(sub[-_]?[A-Za-z0-9]+)", path.stem, flags=re.IGNORECASE)
    return match.group(1).replace("_", "-") if match else path.parent.name

def primary_recording_files(dataset, root):
    files = []
    for path in sorted(root.rglob("*")):
        if not path.is_file():
            continue
        lower_name = path.name.lower()
        if lower_name.endswith(".fif.gz") or path.suffix.lower() in PRIMARY_EXTENSIONS:
            files.append(path)
    return files

recording_rows = []
for dataset, root in DATA_ROOTS.items():
    labels = label_tables[dataset].copy()
    if not labels.empty:
        labels["Subject_ID_key"] = labels["Subject_ID"].astype(str).str.replace("_", "-", regex=False).str.lower()
    label_lookup = labels.drop_duplicates("Subject_ID_key").set_index("Subject_ID_key").to_dict("index") if not labels.empty else {}
    for path in primary_recording_files(dataset, root):
        subject_id = subject_from_filename(dataset, path)
        key = subject_id.lower()
        metadata = label_lookup.get(key, {})
        diagnosis = metadata.get("Diagnosis", "Unknown")
        if dataset == "BrainLat" and diagnosis not in {"AD", "FTD", "HC"}:
            continue
        recording_rows.append({
            "Dataset": dataset,
            "Subject_ID": subject_id,
            "Diagnosis": diagnosis,
            "Source_Label": metadata.get("Source_Label", metadata.get("Group", np.nan)),
            "File_Path": str(path),
            "File_Format": ".fif.gz" if path.name.lower().endswith(".fif.gz") else path.suffix.lower(),
            "Channels": np.nan,
            "Channel_Names": np.nan,
            "Sampling_Rate": np.nan,
            "Duration": np.nan,
            "Read_Status": "not inspected",
            "Read_Error": np.nan,
        })
recordings = pd.DataFrame(recording_rows)
print(f"Primary EEG recordings discovered: {len(recordings)}")
display(recordings.head())

Primary EEG recordings discovered: 310


,Dataset,Subject_ID,Diagnosis,Source_Label,File_Path,File_Format,Channels,Channel_Names,Sampling_Rate,Duration,Read_Status,Read_Error
0,ADFTD,sub-001,AD,A,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/derivatives/sub-001/eeg/sub-001_task-eyesclosed_eeg.set,.set,NaN,NaN,NaN,NaN,not inspected,NaN
1,ADFTD,sub-002,AD,A,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/derivatives/sub-002/eeg/sub-002_task-eyesclosed_eeg.set,.set,NaN,NaN,NaN,NaN,not inspected,NaN
2,ADFTD,sub-003,AD,A,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/derivatives/sub-003/eeg/sub-003_task-eyesclosed_eeg.set,.set,NaN,NaN,NaN,NaN,not inspected,NaN
3,ADFTD,sub-004,AD,A,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/derivatives/sub-004/eeg/sub-004_task-eyesclosed_eeg.set,.set,NaN,NaN,NaN,NaN,not inspected,NaN
4,ADFTD,sub-005,AD,A,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/derivatives/sub-005/eeg/sub-005_task-eyesclosed_eeg.set,.set,NaN,NaN,NaN,NaN,not inspected,NaN


## 4. Inspect EEG headers without loading signal data

The following function reads recording headers with `preload=False`. It extracts channel names, channel count, sampling rate, and duration. It does not load every recording's signal matrix into memory. Failures are preserved as explicit corrupted/unreadable candidates.

In [ ]:
def read_header(path):
    suffix = ".fif.gz" if path.name.lower().endswith(".fif.gz") else path.suffix.lower()
    readers = {
        ".set": lambda: mne.io.read_raw_eeglab(path, preload=False, verbose="ERROR"),
        ".edf": lambda: mne.io.read_raw_edf(path, preload=False, verbose="ERROR"),
        ".bdf": lambda: mne.io.read_raw_bdf(path, preload=False, verbose="ERROR"),
        ".fif": lambda: mne.io.read_raw_fif(path, preload=False, verbose="ERROR"),
        ".fif.gz": lambda: mne.io.read_raw_fif(path, preload=False, verbose="ERROR"),
        ".vhdr": lambda: mne.io.read_raw_brainvision(path, preload=False, verbose="ERROR"),
        ".cnt": lambda: mne.io.read_raw_cnt(path, preload=False, verbose="ERROR"),
    }
    if suffix not in readers:
        raise ValueError(f"No safe header reader configured for {suffix}")
    try:
        raw = readers[suffix]()
    except NotImplementedError:
        sidecar_json = path.with_name(path.name.replace("_eeg.set", "_eeg.json"))
        channels_tsv = path.with_name(path.name.replace("_eeg.set", "_channels.tsv"))
        if not sidecar_json.exists() or not channels_tsv.exists():
            raiseI am building an AI-based dementia screening system using EEG data.

I have downloaded 3 EEG datasets:

1. ADFTD (Alzheimer's Disease, Frontotemporal Dementia, Healthy Controls)
2. BrainLat (AD, bvFTD, Healthy Controls, with other disease folders that should NOT be used)
3. AD-Auditory (AD, MCI, Healthy Controls)

The datasets are stored in:

Dementia_EEG_Project/
└── data/
    ├── ADFTD/
    ├── BrainLat/
    └── AD_Auditory/

I am currently doing DATA INSPECTION ONLY.

DO NOT preprocess, modify, rename, move, delete, or combine any files yet.

Create a Python/Jupyter Notebook called:
01_Dataset_Inspection.ipynb

The notebook must automatically inspect all 3 datasets and generate a clear report.

For each dataset, determine:

1. Dataset name
2. Number of subjects
3. Number of EEG recording files
4. EEG file formats/extensions (EDF, FIF, BDF, BrainVision, etc.)
5. Diagnosis/class labels available
6. Number of subjects in each class
7. EEG channels available
8. Number of channels per recording
9. Sampling frequency (Hz)
10. Recording duration
11. Missing or corrupted EEG files
12. Missing metadata
13. Important metadata files such as participants.tsv, CSV, JSON, etc.
14. Whether the EEG recordings are resting-state or task-based, if this information is available in the metadata
15. Whether the dataset contains MCI, and keep MCI separate from AD
16. For BrainLat, inspect ONLY the 1_AD, 2_bvFTD and 5_HC folders. Ignore 3_PD and 4_MS.

Use Python libraries such as:
- os
- pathlib
- pandas
- numpy
- mne
- json

The notebook should NOT load every large EEG recording into memory unnecessarily. First inspect the directory structure and metadata. For EEG file properties, inspect a representative recording from each subject/dataset where practical.

Create summary tables using pandas.

At the end, generate a final comparison table:

Dataset | Subjects | AD | FTD/bvFTD | HC | MCI | EEG Format | Channels | Sampling Rate

Also create a list of every EEG recording with:

Dataset | Subject_ID | Diagnosis | File_Path | File_Format | Channels | Sampling_Rate | Duration

IMPORTANT:
- Do not randomly assign labels.
- Do not assume a folder name is a diagnosis unless supported by the dataset structure/metadata.
- Preserve the original subject IDs.
- Do not split subjects into train/test yet.
- Do not preprocess or normalize EEG yet.
- Do not combine datasets yet.
- Do not delete MCI.
- Clearly report anything that cannot be determined from the downloaded data.

At the end, print:
"DATA INSPECTION COMPLETE — READY FOR HARMONIZATION"

Make the notebook beginner-friendly with Markdown explanations before each code section.

        with sidecar_json.open(encoding="utf-8") as handle:
            sidecar = json.load(handle)
        channels_table = pd.read_csv(channels_tsv, sep="\t")
        channel_names = channels_table["name"].dropna().astype(str).tolist()
        sfreq = float(sidecar["SamplingFrequency"])
        duration = float(sidecar["RecordingDuration"])
        return len(channel_names), "; ".join(channel_names), sfreq, duration, "metadata_only"
    try:
        sfreq = float(raw.info.get("sfreq", np.nan))
        n_times = int(raw.n_times)
        return len(raw.ch_names), "; ".join(raw.ch_names), sfreq, n_times / sfreq if sfreq else np.nan, "readable"
    finally:
        raw.close()

for index, row in recordings.iterrows():
    path = Path(row["File_Path"])
    try:
        channels, channel_names, sfreq, duration, status = read_header(path)
        recordings.loc[index, ["Channels", "Channel_Names", "Sampling_Rate", "Duration", "Read_Status"]] = [channels, channel_names, sfreq, duration, status]
    except Exception as exc:
        recordings.loc[index, "Read_Status"] = "failed"
        recordings.loc[index, "Read_Error"] = f"{type(exc).__name__}: {exc}"

recordings["Channels"] = pd.to_numeric(recordings["Channels"], errors="coerce").astype("Int64")
recordings["Sampling_Rate"] = pd.to_numeric(recordings["Sampling_Rate"], errors="coerce")
recordings["Duration"] = pd.to_numeric(recordings["Duration"], errors="coerce")
display(recordings[["Dataset", "Subject_ID", "Diagnosis", "File_Path", "File_Format", "Channels", "Sampling_Rate", "Duration", "Read_Status"]])

failed_recordings = recordings[recordings["Read_Status"] == "failed"]
print(f"Unreadable or corrupted candidate recordings: {len(failed_recordings)}")
display(failed_recordings[["Dataset", "Subject_ID", "File_Path", "Read_Error"]])

,Dataset,Subject_ID,Diagnosis,File_Path,File_Format,Channels,Sampling_Rate,Duration,Read_Status
0,ADFTD,sub-001,AD,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/derivatives/sub-001/eeg/sub-001_task-eyesclosed_eeg.set,.set,19,500.0,599.8,readable
1,ADFTD,sub-002,AD,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/derivatives/sub-002/eeg/sub-002_task-eyesclosed_eeg.set,.set,19,500.0,793.1,readable
2,ADFTD,sub-003,AD,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/derivatives/sub-003/eeg/sub-003_task-eyesclosed_eeg.set,.set,19,500.0,306.1,readable
3,ADFTD,sub-004,AD,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/derivatives/sub-004/eeg/sub-004_task-eyesclosed_eeg.set,.set,19,500.0,706.1,readable
4,ADFTD,sub-005,AD,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/derivatives/sub-005/eeg/sub-005_task-eyesclosed_eeg.set,.set,19,500.0,804.1,readable
...,...,...,...,...,...,...,...,...,...
305,AD_Auditory,sub-31,HC,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt 3/sub-31/eeg/sub-31_task-40HzAuditoryEntrainment_eeg.set,.set,19,250.0,590.0,metadata_only
306,AD_Auditory,sub-32,MCI,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt 3/sub-32/eeg/sub-32_task-40HzAuditoryEntrainment_eeg.set,.set,19,250.0,590.0,metadata_only
307,AD_Auditory,sub-33,AD,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt 3/sub-33/eeg/sub-33_task-40HzAuditoryEntrainment_eeg.set,.set,19,250.0,590.0,metadata_only
308,AD_Auditory,sub-34,AD,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt 3/sub-34/eeg/sub-34_task-40HzAuditoryEntrainment_eeg.set,.set,19,250.0,590.0,metadata_only


Unreadable or corrupted candidate recordings: 0


,Dataset,Subject_ID,File_Path,Read_Error


## 5. Missing metadata, labels, task/rest status, and BrainLat exclusions

This section compares metadata subjects with recording subjects, reports missing label matches, identifies missing sidecars, and uses explicit metadata fields such as `Rest` or BIDS task names when available. A task name such as `eyesclosed` or `40HzAuditoryEntrainment` is reported as task/context metadata; it is not silently reclassified as resting state.

In [20]:
def metadata_status(dataset, row):
    root = DATA_ROOTS[dataset]
    path = Path(row["File_Path"])
    sidecars = []
    if path.suffix.lower() == ".set":
        sidecars = [path.with_suffix(".json"), path.with_suffix(".fdt")]
    elif path.suffix.lower() in {".edf", ".bdf", ".fif", ".vhdr"}:
        sidecars = [path.with_suffix(".json"), path.with_suffix(".tsv")]
    existing = [str(item.relative_to(root)) for item in sidecars if item.exists()]
    missing = [str(item.relative_to(root)) for item in sidecars if not item.exists()]
    return existing, missing

recordings[["Present_Sidecars", "Missing_Sidecars"]] = recordings.apply(lambda row: pd.Series(metadata_status(row["Dataset"], row)), axis=1)
recordings["Study_Context"] = recordings["File_Path"].map(lambda value: "eyes-closed / likely resting context from task name" if "eyesclosed" in value.lower() else ("auditory 40 Hz task from BIDS task name" if "40hz" in value.lower() else ("resting-state indicated by filename" if "_rs" in value.lower() else "not determined from filename")))

missing_rows = []
for dataset, labels in label_tables.items():
    metadata_subjects = set(labels["Subject_ID"].astype(str)) if not labels.empty else set()
    recording_subjects = set(recordings.loc[recordings["Dataset"] == dataset, "Subject_ID"].astype(str))
    missing_rows.append({"Dataset": dataset, "Metadata subjects": len(metadata_subjects), "Recording subjects": len(recording_subjects), "Metadata without recording": len(metadata_subjects - recording_subjects), "Recording without metadata": len(recording_subjects - metadata_subjects)})
missing_metadata_table = pd.DataFrame(missing_rows)
display(missing_metadata_table)

print("Unknown or unmapped recording labels:")
display(recordings[recordings["Diagnosis"] == "Unknown"][["Dataset", "Subject_ID", "Source_Label", "File_Path"]])

brainlat_source_labels = sorted(set(brainlat_labels.get("diagnosis", pd.Series(dtype=str)).dropna().astype(str)))
print("BrainLat labels observed in metadata:", brainlat_source_labels)
print("BrainLat included labels: AD, FTD/bvFTD, CN/HC. Excluded labels: PD and MS.")

print("Missing sidecar/metadata examples:")
display(recordings[recordings["Missing_Sidecars"].map(bool)][["Dataset", "Subject_ID", "File_Path", "Missing_Sidecars"]].head(30))

,Dataset,Metadata subjects,Recording subjects,Metadata without recording,Recording without metadata
0,ADFTD,88,88,0,0
1,AD_Auditory,35,35,0,0
2,BrainLat,99,99,0,0


Unknown or unmapped recording labels:


,Dataset,Subject_ID,Source_Label,File_Path
280,AD_Auditory,sub-06,-,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt 3/sub-06/eeg/sub-06_task-40HzAuditoryEntrainment_eeg.set
287,AD_Auditory,sub-13,-,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt 3/sub-13/eeg/sub-13_task-40HzAuditoryEntrainment_eeg.set
298,AD_Auditory,sub-24,Moderate AD,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt 3/sub-24/eeg/sub-24_task-40HzAuditoryEntrainment_eeg.set


BrainLat labels observed in metadata: ['AD', 'CN', 'FTD']
BrainLat included labels: AD, FTD/bvFTD, CN/HC. Excluded labels: PD and MS.
Missing sidecar/metadata examples:


,Dataset,Subject_ID,File_Path,Missing_Sidecars
0,ADFTD,sub-001,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/derivatives/sub-001/eeg/sub-001_task-eyesclosed_eeg.set,"[derivatives/sub-001/eeg/sub-001_task-eyesclosed_eeg.json, derivatives/sub-001/eeg/sub-001_task-eyesclosed_eeg.fdt]"
1,ADFTD,sub-002,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/derivatives/sub-002/eeg/sub-002_task-eyesclosed_eeg.set,"[derivatives/sub-002/eeg/sub-002_task-eyesclosed_eeg.json, derivatives/sub-002/eeg/sub-002_task-eyesclosed_eeg.fdt]"
2,ADFTD,sub-003,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/derivatives/sub-003/eeg/sub-003_task-eyesclosed_eeg.set,"[derivatives/sub-003/eeg/sub-003_task-eyesclosed_eeg.json, derivatives/sub-003/eeg/sub-003_task-eyesclosed_eeg.fdt]"
3,ADFTD,sub-004,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/derivatives/sub-004/eeg/sub-004_task-eyesclosed_eeg.set,"[derivatives/sub-004/eeg/sub-004_task-eyesclosed_eeg.json, derivatives/sub-004/eeg/sub-004_task-eyesclosed_eeg.fdt]"
4,ADFTD,sub-005,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/derivatives/sub-005/eeg/sub-005_task-eyesclosed_eeg.set,"[derivatives/sub-005/eeg/sub-005_task-eyesclosed_eeg.json, derivatives/sub-005/eeg/sub-005_task-eyesclosed_eeg.fdt]"
5,ADFTD,sub-006,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/derivatives/sub-006/eeg/sub-006_task-eyesclosed_eeg.set,"[derivatives/sub-006/eeg/sub-006_task-eyesclosed_eeg.json, derivatives/sub-006/eeg/sub-006_task-eyesclosed_eeg.fdt]"
6,ADFTD,sub-007,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/derivatives/sub-007/eeg/sub-007_task-eyesclosed_eeg.set,"[derivatives/sub-007/eeg/sub-007_task-eyesclosed_eeg.json, derivatives/sub-007/eeg/sub-007_task-eyesclosed_eeg.fdt]"
7,ADFTD,sub-008,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/derivatives/sub-008/eeg/sub-008_task-eyesclosed_eeg.set,"[derivatives/sub-008/eeg/sub-008_task-eyesclosed_eeg.json, derivatives/sub-008/eeg/sub-008_task-eyesclosed_eeg.fdt]"
8,ADFTD,sub-009,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/derivatives/sub-009/eeg/sub-009_task-eyesclosed_eeg.set,"[derivatives/sub-009/eeg/sub-009_task-eyesclosed_eeg.json, derivatives/sub-009/eeg/sub-009_task-eyesclosed_eeg.fdt]"
9,ADFTD,sub-010,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/derivatives/sub-010/eeg/sub-010_task-eyesclosed_eeg.set,"[derivatives/sub-010/eeg/sub-010_task-eyesclosed_eeg.json, derivatives/sub-010/eeg/sub-010_task-eyesclosed_eeg.fdt]"


## 6. Dataset summaries

The summary tables below distinguish subject counts from recording counts and keep MCI separate from AD. Channel names are shown as a union of names found in readable headers, while the number of channels is summarized as observed values rather than forced into a single assumed value.

In [21]:
def joined_unique(values):
    items = []
    for value in values.dropna():
        items.extend([item.strip() for item in str(value).split(";") if item.strip()])
    return ", ".join(sorted(set(items))) if items else "Undetermined"

summary_rows = []
for dataset in DATA_ROOTS:
    subset = recordings[recordings["Dataset"] == dataset]
    readable = subset[subset["Read_Status"].isin({"readable", "metadata_only"})]
    subjects = set(subset["Subject_ID"].dropna())
    class_counts = subset.groupby("Diagnosis")["Subject_ID"].nunique().to_dict()
    formats = ", ".join(sorted(subset["File_Format"].dropna().unique())) or "Undetermined"
    rates = ", ".join(sorted({f"{value:g}" for value in readable["Sampling_Rate"].dropna()})) or "Undetermined"
    channel_counts = ", ".join(sorted({str(int(value)) for value in readable["Channels"].dropna()})) or "Undetermined"
    summary_rows.append({
        "Dataset": dataset,
        "Subjects": len(subjects),
        "EEG recordings": len(subset),
        "AD": class_counts.get("AD", 0),
        "FTD/bvFTD": class_counts.get("FTD", 0),
        "HC": class_counts.get("HC", 0),
        "MCI": class_counts.get("MCI", 0),
        "Unknown": class_counts.get("Unknown", 0),
        "EEG Format": formats,
        "Channels": channel_counts,
        "Sampling Rate": rates,
        "Rest/task evidence": ", ".join(sorted(subset["Study_Context"].dropna().unique())) or "Undetermined",
    })
dataset_summary = pd.DataFrame(summary_rows)
display(dataset_summary)

print("Channel names observed in readable headers by dataset:")
for dataset in DATA_ROOTS:
    channel_values = recordings.loc[recordings["Dataset"] == dataset, "Channel_Names"]
    print(f"\n{dataset}: {joined_unique(channel_values)}")

,Dataset,Subjects,EEG recordings,AD,FTD/bvFTD,HC,MCI,Unknown,EEG Format,Channels,Sampling Rate,Rest/task evidence
0,ADFTD,88,176,36,23,29,0,0,.set,19,500,eyes-closed / likely resting context from task name
1,BrainLat,99,99,35,18,46,0,0,.set,128,512,resting-state indicated by filename
2,AD_Auditory,35,35,16,0,10,6,3,.set,19,250,auditory 40 Hz task from BIDS task name


Channel names observed in readable headers by dataset:

ADFTD: C3, C4, Cz, F3, F4, F7, F8, Fp1, Fp2, Fz, O1, O2, P3, P4, Pz, T3, T4, T5, T6

BrainLat: A1, A10, A11, A12, A13, A14, A15, A16, A17, A18, A19, A2, A20, A21, A22, A23, A24, A25, A26, A27, A28, A29, A3, A30, A31, A32, A4, A5, A6, A7, A8, A9, B1, B10, B11, B12, B13, B14, B15, B16, B17, B18, B19, B2, B20, B21, B22, B23, B24, B25, B26, B27, B28, B29, B3, B30, B31, B32, B4, B5, B6, B7, B8, B9, C1, C10, C11, C12, C13, C14, C15, C16, C17, C18, C19, C2, C20, C21, C22, C23, C24, C25, C26, C27, C28, C29, C3, C30, C31, C32, C4, C5, C6, C7, C8, C9, D1, D10, D11, D12, D13, D14, D15, D16, D17, D18, D19, D2, D20, D21, D22, D23, D24, D25, D26, D27, D28, D29, D3, D30, D31, D32, D4, D5, D6, D7, D8, D9

AD_Auditory: C3, C4, Cz, F3, F4, F7, F8, Fp1, Fp2, Fz, O1, O2, P3, P4, P7, P8, Pz, T7, T8


## 7. Required final comparison and recording-level tables

These are the compact outputs intended for review before harmonization. The recording table preserves original subject identifiers and file paths. No train/test split or cross-dataset combination is performed.

In [22]:
comparison_columns = ["Dataset", "Subjects", "AD", "FTD/bvFTD", "HC", "MCI", "EEG Format", "Channels", "Sampling Rate"]
final_comparison = dataset_summary[comparison_columns].copy()
print("FINAL COMPARISON TABLE")
display(final_comparison)

recording_columns = ["Dataset", "Subject_ID", "Diagnosis", "File_Path", "File_Format", "Channels", "Sampling_Rate", "Duration"]
recording_inventory = recordings[recording_columns].sort_values(["Dataset", "Subject_ID", "File_Path"]).reset_index(drop=True)
print("COMPLETE EEG RECORDING INVENTORY")
display(recording_inventory)

print("Important limitations to resolve before harmonization:")
limitations = []
if len(failed_recordings):
    limitations.append(f"{len(failed_recordings)} recording(s) could not be read from headers.")
if (recordings["Diagnosis"] == "Unknown").any():
    limitations.append("Some recordings have no metadata-supported diagnosis match.")
if recordings["Sampling_Rate"].isna().any():
    limitations.append("Sampling rate is unavailable for at least one recording.")
if not limitations:
    limitations.append("No limitations were detected by the checks above; confirm the displayed tables manually.")
for item in limitations:
    print("-", item)

print("DATA INSPECTION COMPLETE — READY FOR HARMONIZATION")

FINAL COMPARISON TABLE


,Dataset,Subjects,AD,FTD/bvFTD,HC,MCI,EEG Format,Channels,Sampling Rate
0,ADFTD,88,36,23,29,0,.set,19,500
1,BrainLat,99,35,18,46,0,.set,128,512
2,AD_Auditory,35,16,0,10,6,.set,19,250


COMPLETE EEG RECORDING INVENTORY


,Dataset,Subject_ID,Diagnosis,File_Path,File_Format,Channels,Sampling_Rate,Duration
0,ADFTD,sub-001,AD,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/derivatives/sub-001/eeg/sub-001_task-eyesclosed_eeg.set,.set,19,500.0,599.800000
1,ADFTD,sub-001,AD,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/sub-001/eeg/sub-001_task-eyesclosed_eeg.set,.set,19,500.0,599.800000
2,ADFTD,sub-002,AD,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/derivatives/sub-002/eeg/sub-002_task-eyesclosed_eeg.set,.set,19,500.0,793.100000
3,ADFTD,sub-002,AD,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/sub-002/eeg/sub-002_task-eyesclosed_eeg.set,.set,19,500.0,793.100000
4,ADFTD,sub-003,AD,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt1/derivatives/sub-003/eeg/sub-003_task-eyesclosed_eeg.set,.set,19,500.0,306.100000
...,...,...,...,...,...,...,...,...
305,BrainLat,sub-30031,AD,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt 2/s6_sub-30031_rs-HEP_eeg.set,.set,128,512.0,412.601562
306,BrainLat,sub-30032,AD,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt 2/s6_sub-30032_rs-HEP_eeg.set,.set,128,512.0,560.277344
307,BrainLat,sub-30033,AD,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt 2/s6_sub-30033_rs-HEP_eeg.set,.set,128,512.0,498.896484
308,BrainLat,sub-30034,AD,/home/admin1/Desktop/SIH_2026_ALL/SIH_Data/dt 2/s6_sub-30034_rs-HEP_eeg.set,.set,128,512.0,580.214844


Important limitations to resolve before harmonization:
- Some recordings have no metadata-supported diagnosis match.
DATA INSPECTION COMPLETE — READY FOR HARMONIZATION
